In [6]:
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(
    model='gemini-1.5-pro',
    temperature=0.7,
    max_tokens=None,
    timeout=None,
    max_retries=3,
    
)


In [5]:
print(os.getenv("SERPAPI_API_KEY"))

ba0e5b63ea224811ce8950ff1f194cc2dfdd909f18700df721535c127db1553c


## create the google seach tools

In [9]:
from langchain.utilities import SerpAPIWrapper
from langchain.tools import Tool
def get_serpapi_key():
    return os.getenv("SERPAPI_API_KEY")

def create_serpapi_search():
    search = SerpAPIWrapper(serpapi_api_key=get_serpapi_key())
    return search

def create_google_search_tool():
    search = create_serpapi_search()
    tool = Tool(
        name="Google Search",
        description="Useful for answering questions by searching Google.",
        func=search.run
    )
    return tool

google_search_tool = create_google_search_tool()

query = "Today's IPL Live Score?"
result = google_search_tool.run(query)

print("\nResult:\n", result)




Result:
 {'title': 'Indian Premier League', 'thumbnail': 'https://serpapi.com/searches/680e2c244cace737cc69a5f3/images/fe8e11684ea81a2ddec0876bac7416ca8b6bf26d7f8058466fe4e042e8dbdba6.png', 'game_spotlight': {'league': 'IPL', 'stadium': 'Wankhede Stadium', 'stadium_kgmid': '/m/05f74n', 'stage': 'T20 45 of 74', 'date': 'today, 5:00\u202fAM', 'status': 'Live', 'video_highlight_carousel': [{'title': 'Milestone wicket, excellent over: Jasprit Bumrah strikes early', 'link': 'https://www.iplt20.com/video/61853/milestone-wicket-excellent-over-jasprit-bumrah-strikes-early?tagNames=2025?utm_source=video&utm_medium=onebox&utm_campaign=ipl2025', 'duration': '1:01', 'thumbnail': 'https://ssl.gstatic.com/onebox/media/sports/videos/vita/3kEJ0gR-IHv-FLsJ_768x432.jpg'}, {'title': 'Debutant Bosch, Dynamic Dhir provide fabulous final flourish', 'link': 'https://www.iplt20.com/video/61848/debutant-bosch-dynamic-dhir-provide-fabulous-final-flourish?tagNames=2025?utm_source=video&utm_medium=onebox&utm_cam

## Create weather info tool

In [18]:
#!pip install  requests
import requests
def get_weather_api():
    return os.getenv("OPENWEATHERMAP_API_KEY")

#weather search function
def weather_info(city_name):
    base_url = "https://api.openweathermap.org/data/2.5/weather"
    params = {
        "q": city_name,
        "appid": get_weather_api(),
        "units": "metric"  # for temperature in Celsius
    }
    response = requests.get(base_url, params=params)
    
    if response.status_code != 200:
        return f"Error fetching weather: {response.text}"
    
    data = response.json()
    temp = data["main"]["temp"]
    description = data["weather"][0]["description"]
    return f"The weather in {city_name} is {description} with a temperature of {temp}°C."

#create tool and pass function
def create_weather_tool():
    tool = Tool(
        name="Weather Checker",
        description="Useful for checking the current weather in a city. Input should be a city name.",
        func=weather_info
    )
    return tool

# now tool created ready to consume

#weather_tool = create_weather_tool()

query = "Antarctica"
result = create_weather_tool().run(query)

print("\nWeather Result:\n", result)



Weather Result:
 The weather in Antarctica is broken clouds with a temperature of -51.71°C.


In [2]:
## Cricker live match API

In [7]:
# Import modules
import requests
import os
from langchain.tools import Tool

# Config - Cricket API Key
def get_cricket_api_key():
    return os.getenv("Cricket_API")


# Fetch live cricket score
def get_live_cricket_score():
    api_key = get_cricket_api_key()
    print(api_key)
    url = f"https://api.cricapi.com/v1/currentMatches?apikey={api_key}&offset=0"

    response = requests.get(url)
    if response.status_code != 200:
        return f"Error fetching score: {response.text}"

    data = response.json()
    print(data)

    if not data.get("data"):
        return "No live matches found."

    results = []
    for match in data["data"]:
        if match["status"] == "live":
            team1 = match["teams"][0]
            team2 = match["teams"][1]
            score = match.get("score", [])
            status = match.get("status", "Unknown")

            results.append(f"{team1} vs {team2} - Status: {status}")

    if not results:
        return "No ongoing live matches at the moment."

    return "\n".join(results)

# Create Cricket Score Tool
def create_cricket_score_tool():
    tool = Tool(
        name="Live Cricket Score Checker",
        description="Useful for checking live cricket match scores. No input required.",
        func=lambda x: get_live_cricket_score()
    )
    return tool

# Use it!

cricket_tool = create_cricket_score_tool()

query = ""  # No input needed
result = cricket_tool.run(query)

print("\nCricket Score Result:\n", result)



8713a673-15b4-4447-a649-e62938d4c0d9
{'apikey': '8713a673-15b4-4447-a649-e62938d4c0d9', 'data': [{'id': 'd1d4777a-ab8d-4d5b-bd44-14faeafacc76', 'name': 'United States of America Women vs Zimbabwe Women, 2nd T20I', 'matchType': 't20', 'status': 'Zimbabwe Women won by 1 run', 'venue': 'Grand Prairie Stadium, Dallas', 'date': '2025-04-27', 'dateTimeGMT': '2025-04-27T15:45:00', 'teams': ['United States of America Women', 'Zimbabwe Women'], 'score': [{'r': 136, 'w': 5, 'o': 20, 'inning': 'Zimbabwe Women Inning 1'}, {'r': 135, 'w': 6, 'o': 20, 'inning': 'United States of America Women Inning 1'}], 'series_id': '57991bad-36cd-46f1-b3f7-641483b8e8df', 'fantasyEnabled': True, 'bbbEnabled': True, 'hasSquad': True, 'matchStarted': True, 'matchEnded': True}, {'id': 'a1821ef7-792a-4c5d-9fec-53b870848a39', 'name': 'Malaysia vs Singapore, 8th Match', 'matchType': 't20', 'status': 'No result due to rain', 'venue': 'Bayuemas Oval, Kuala Lumpur', 'date': '2025-04-28', 'dateTimeGMT': '2025-04-28T06:00:00

Here similarly I can create multiple tools as per need such as 
Football Score Tool
Stock Market Price Tool
Bitcoin Price Checker
Air Quality API Tool
News API Tool

Process:
create function that return api
create fultion that consume that API will do  tools task such as live score, stock market price etc
here one might need to do certain parsing it is very subjective to task what and how we have to do.
tip:- first print the whole output if after getting status code response 200
      how to use api check documentation and example availabe for majority of them

once it is done create the wrapper of tool and one can use @tool decorator
You can use above example as template
@tool is more useful with agent as per my understanding